# Prompt Parsing
These are library test scripts, small code samples, and larger systems all designed around the prompt parsing and user interface aspect of this algorithm.

## Basic Snippets

### Basic `spaCy` Test
This script accepts user input and runs the `spaCy` model on it, tokenizing the input and printing out each token and its detected part of speech.

**Notes**
1. If the input is "dog red", red is parsed as a noun. But if the input is "red dog", red is parsed as an adjective

In [10]:
import spacy
from spacy.lang.en.examples import sentences 

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)
for token in doc:
    print(token.text, " | ", spacy.explain(token.pos_))

10 year old boy in red winter clothing
10  |  numeral
year  |  noun
old  |  adjective
boy  |  noun
in  |  adposition
red  |  adjective
winter  |  noun
clothing  |  noun


### Noun Filtering
This script accepts user input and after tokenizing it, prints out the detected nouns.

In [1]:
import spacy
from spacy.lang.en.examples import sentences 

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

nouns = []
for token in doc:
    if(spacy.explain(token.pos_) == "noun"):
        nouns.append(token.text)

for noun in nouns:
    print(noun)

10 year old boy in red winter clothing
year
boy
winter
clothing


### Adjective Filtering
This script accepts user input and after tokenizing it, prints out the detected adjectives.

In [11]:
import spacy
from spacy.lang.en.examples import sentences 

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

adjectives = []
for token in doc:
    if(spacy.explain(token.pos_) == "adjective"):
        adjectives.append(token.text)

for adj in adjectives:
    print(adj)

10 year old boy in red winter clothing
old
red


### Parsing Multiple Parts of Speech
This script accepts user input and after tokenizing it, prints out each word/token and its detected POS.

In [ ]:
import spacy

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

tokens = []

for token in doc:
    tokens.append({"text": token.text, "type": token.pos_})

for token in tokens:
    print(token["text"] + " - " + token["type"])

10 year old boy in red winter clothing
10- NUM
year- NOUN
old- ADJ
boy- NOUN
in- ADP
red- ADJ
winter- NOUN
clothing- NOUN


# Stage 1: Dependency Parsing
The first stage of this algorithm involves parsing the prompt the user gives. Previously, I looked at the basics of using the `spaCy` library and used it to parse the prompt into each word's part of speech.

This isn't particularly useful though because it doesn't tell us anything about each word. So we add dependency parsing which uses `spaCy` to figure out which word the current word is related to. This can help make relations and figure out what the prompt actually means.

## Basic Parsing
This script lists out each token and what word it relates to. 

In [6]:
import spacy

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

tokens = []

for token in doc:
    tokens.append({"text": token.text, "type": spacy.explain(token.dep_), "dependency": token.head.text})

for token in tokens:
    print(f"{token["text"]} is a {token["type"]} and relates to {token["dependency"]}")

10 year old boy in red winter clothing lost in a dense forest
10 is a numeric modifier and relates to year
year is a noun phrase as adverbial modifier and relates to old
old is a adjectival modifier and relates to boy
boy is a nominal subject and relates to lost
in is a prepositional modifier and relates to boy
red is a adjectival modifier and relates to winter
winter is a compound and relates to clothing
clothing is a object of preposition and relates to in
lost is a root and relates to lost
in is a prepositional modifier and relates to lost
a is a determiner and relates to forest
dense is a adjectival modifier and relates to forest
forest is a object of preposition and relates to in


## Semantic Slots
Now we define certain slots that we would sort the tokens into for most prompts. This includes the main search object/subject, situations/verbs, physical descriptions, context/environmental clues, etc.

As you can see, this naive algorithm doesn't work quite as expected.

In [11]:
import spacy

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

searchObject = []
situation = []
descriptions = []

print(doc.text)

for token in doc:
    if(spacy.explain(token.dep_) == "root"):
        searchObject.append(token.text)
    elif(spacy.explain(token.pos_) == "verb"):
        situation.append(token.text)
    elif(spacy.explain(token.pos_) == "adjective"):
        descriptions.append(token.text)

for obj in searchObject:
    print("OBJECT: ", obj)

for verb in situation:
    print("SITUATION: ", verb)

for adj in descriptions:
   print("DESCRIPTION: ", adj) 

10 year old boy in red winter clothing lost in the dense forest
OBJECT:  lost
DESCRIPTION:  old
DESCRIPTION:  red
DESCRIPTION:  dense


### A little help?
I wasn't sure where to go from here especially once I realized that not every prompt will have the same word types. Which means that even if I was able to figure out how to parse my test prompt, it would almost definitely not work for other sentence structures.

So I asked ChatGPT 5.2 the following prompt (actually a combination of multiple prompts, each with varying levels of success) and the following code block is the code that it gave me.

Note: I added a parsing function which wraps ChatGPT's code to allow for second prompt to be parsed at the same time to show how this algorithm doesn't work

> so i'm working on prompt parsing right now and i have a very basic snippet worked out that can take a prompt and spit out each word and its part of speech, what do i do now?

> help me understand what i need to do to parse this prompt: "10 year old boy in red winter clothing lost in the dense forest"

> give me code that can parse it exactly like that

In [33]:
import spacy

nlp = spacy.load("en_core_web_sm")

# This function's code is all from ChatGPT 
def parse():
    doc = nlp(prompt)

    result = {
        "target_entity": None,
        "age": None,
        "clothing": [],
        "state": None,
        "environment": {}
    }

    # --- 1. find state (ROOT verb or adjectival verb) ---
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ in ("VERB", "ADJ"):
            result["state"] = token.text
            state_token = token
            break

    # --- 2. find target entity (subject of state) ---
    for token in doc:
        if token.dep_ == "nsubj" and token.head == state_token:
            result["target_entity"] = token.text
            target = token
            break

    # --- 3. extract age ---
    for token in doc:
        if token.dep_ == "nummod" and token.head.text in ("year", "years"):
            # walk upward until boy
            head = token.head
            while head.head != head:
                if head.head == target:
                    result["age"] = token.text
                    break
                head = head.head

    # --- 4. extract clothing ---
    for token in doc:
        if token.text == "clothing" and token.dep_ == "pobj":
            for child in token.children:
                if child.dep_ in ("amod", "compound"):
                    result["clothing"].append(child.text)

    # --- 5. extract environment ---
    for token in doc:
        if token.dep_ == "pobj" and token.text == "forest":
            result["environment"]["type"] = token.text
            for child in token.children:
                if child.dep_ == "amod":
                    result["environment"]["descriptor"] = child.text

    print(result)

# Works
prompt = "10 year old boy in red winter clothing lost in the dense forest"
parse()

# Does NOT work
prompt = "small child wearing a blue jacket missing in the park"
parse()

{'target_entity': 'boy', 'age': '10', 'clothing': ['winter'], 'state': 'lost', 'environment': {'type': 'forest', 'descriptor': 'dense'}}
{'target_entity': None, 'age': None, 'clothing': [], 'state': None, 'environment': {}}


## Dependency Tree Testing
To help figure out which direction my algorithm needs to go in, I will be printing out **everything** I believe is useful information from the `spaCy` document and hopefully, I will see a pattern I can pursue.

In [21]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

def parse(prompt):
    promptDoc = nlp(prompt)

    print("| {0:>10} | {1:>10} | {2:>10} | {3:>10} | {4:>10} | {5:>10} |".format("Text", "POS", "Tag", "Dep", "Head", "Is Stop"))
    for token in promptDoc:
        print("| {0:>10} | {1:>10} | {2:>10} | {3:>10} | {4:>10} | {5:>10} |".format(token.text, token.pos_, token.tag_, token.dep_, token.head.text, token.is_stop))
    
    displacy.render(promptDoc, style="dep")

#parse("10 year old boy in red winter clothing lost in the dense forest")
# Added later for testing [identifying the search subject]
#parse("lost child in the north section of the dense pine forest")
#parse("there is something missing somewhere")

# Added later for testing [identifying states]
# parse("gray backpack lost in aisle C")
# parse("gray backpack in aisle C")
# parse("lost boy, 10 years old, wearing red winter clothing in a forest")
# parse("small child wearing a blue jacket missing in the park")

# Added later for testing [identifying descriptions]
parse("10 year old boy in red winter clothing lost in the dense forest")
parse("small child wearing a blue jacket missing in the park")
parse("plastic elephant toy found in aisle B")

|       Text |        POS |        Tag |        Dep |       Head |    Is Stop |
|         10 |        NUM |         CD |     nummod |       year |          0 |
|       year |       NOUN |         NN |   npadvmod |        old |          0 |
|        old |        ADJ |         JJ |       amod |        boy |          0 |
|        boy |       NOUN |         NN |      nsubj |       lost |          0 |
|         in |        ADP |         IN |       prep |        boy |          1 |
|        red |        ADJ |         JJ |       amod |     winter |          0 |
|     winter |       NOUN |         NN |   compound |   clothing |          0 |
|   clothing |       NOUN |         NN |       pobj |         in |          0 |
|       lost |       VERB |        VBN |       ROOT |       lost |          0 |
|         in |        ADP |         IN |       prep |       lost |          1 |
|        the |        DET |         DT |        det |     forest |          1 |
|      dense |        ADJ |         JJ |

|       Text |        POS |        Tag |        Dep |       Head |    Is Stop |
|      small |        ADJ |         JJ |       amod |      child |          0 |
|      child |       NOUN |         NN |       ROOT |      child |          0 |
|    wearing |       VERB |        VBG |        acl |      child |          0 |
|          a |        DET |         DT |        det |     jacket |          1 |
|       blue |        ADJ |         JJ |       amod |     jacket |          0 |
|     jacket |       NOUN |         NN |       dobj |    wearing |          0 |
|    missing |       VERB |        VBG |        acl |     jacket |          0 |
|         in |        ADP |         IN |       prep |    missing |          1 |
|        the |        DET |         DT |        det |       park |          1 |
|       park |       NOUN |         NN |       pobj |         in |          0 |


|       Text |        POS |        Tag |        Dep |       Head |    Is Stop |
|    plastic |      PROPN |        NNP |   compound |        toy |          0 |
|   elephant |       NOUN |         NN |   compound |        toy |          0 |
|        toy |       NOUN |         NN |       ROOT |        toy |          0 |
|      found |       VERB |        VBN |        acl |        toy |          0 |
|         in |        ADP |         IN |       prep |      found |          1 |
|      aisle |       NOUN |         NN |   compound |          B |          0 |
|          B |      PROPN |        NNP |       pobj |         in |          0 |


### Identifying the search subject
From this (mostly the image to be honest), I can see that the verb in the sentence, "lost", is considered the root and not the actual subject/the word we want, "boy". But like ChatGPT had previously mentioned, the relationship between the "root" and the subject we want is that of a "nominal subject"/"nsubj". Basically, in our example, "boy" is the nominal subject of "lost". 

Through looking at the docs, I found out that the word we want could also be "nominal subject passive"/"nsubjpass" so I'll check for that too. Hopefully, this can accurately and consistently return the subject of the sentence that we want. 

In [15]:
import spacy

nlp = spacy.load("en_core_web_sm")

def parse(prompt):
    print(prompt)
    promptDoc = nlp(prompt)
    subject = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence that isn't the subject we want
            for tok in token.children:
                if(tok.pos_ == "NOUN" and tok.dep_ in ["nsubj", "nsubjpass", "dobj"]):
                    subject = tok.text
        elif(token.dep_ == "ROOT"):
            # We have found the noun root of the sentence
            subject = token.text
    print("  Subject: ", subject)

parse("10 year old boy in red winter clothing lost in the dense forest")
parse("10 year old boy in red winter clothing")
parse("small child wearing a blue jacket missing in the park")
parse("gray backpack lost in aisle C")
parse("plastic elephant toy found in aisle B")
# This one didn't work at first so after testing, I added "dobj"/"direct object"s to the possible dependencies
parse("lost child in the north section of the dense pine forest")
parse("object missing at the back of aisle 12")
# This one shouldn't work anyways
parse("there is something missing somewhere")
parse("lost boy, 10 years old, wearing red winter clothing in a forest")


10 year old boy in red winter clothing lost in the dense forest
  Subject:  boy
10 year old boy in red winter clothing
  Subject:  boy
small child wearing a blue jacket missing in the park
  Subject:  child
gray backpack lost in aisle C
  Subject:  backpack
plastic elephant toy found in aisle B
  Subject:  toy
lost child in the north section of the dense pine forest
  Subject:  child
object missing at the back of aisle 12
  Subject:  object
there is something missing somewhere
  Subject:  
lost boy, 10 years old, wearing red winter clothing in a forest
  Subject:  boy


### Identifying states
Okay so identifying the subject of the prompt seems to be working fine. Next, I want to be able to identify the state of the subject. What is happening/happened to it? In our running example with the lost boy in red clothing, the subject is the boy and the state is that he is lost.

Here, we know that the root of the sentence is "lost" so picking it out is easy enough. But in my earliest example: "10 year old boy in red winter clothing", the root word was "boy". My algorithm would still pick "boy" out as the object to search for but there needs to be a way so that the state is an optionally known value and it cannot just be the root of the sentence. 

My thinking is that the state must be the root and a verb but not sure how this will work when the only state is a helping verb/preposition like in the sentence "gray backpack in aisle C" where the state is "in" but it is a preposition.

I'm going to test how it works if we make the state the root verb or if that doesn't exist, a preposition.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def parse(prompt):
    print(prompt)
    promptDoc = nlp(prompt)
    state = ""
    possibleState = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
        elif(token.pos_ == "ADP"):
            # An adpositional phrase includes prepositional phrases, postpositional phrases, and circumpositional phrases
            possibleState = token.text
    if (state == ""):
        state = possibleState
    print("  State: ", state)

parse("10 year old boy in red winter clothing lost in the dense forest")
parse("10 year old boy in red winter clothing")
# This didn't work: missing is not considered a root
parse("small child wearing a blue jacket missing in the park")
# This didn't work: lost is not considered a root
parse("gray backpack lost in aisle C")
# I guess this works because it says that the toy was in aisle B but it was also found? Algorithm should clarify what the state is
parse("plastic elephant toy found in aisle B")
parse("lost child in the north section of the dense pine forest")
# This didn't work: missing is not considered a root
parse("object missing at the back of aisle 12")
# This one shouldn't work anyways but it actually does here
parse("there is something missing somewhere")
# This didn't work: lost is not considered a root 
parse("lost boy, 10 years old, wearing red winter clothing in a forest")


10 year old boy in red winter clothing lost in the dense forest
  State:  lost
10 year old boy in red winter clothing
  State:  in
small child wearing a blue jacket missing in the park
  State:  in
gray backpack lost in aisle C
  State:  in
plastic elephant toy found in aisle B
  State:  in
lost child in the north section of the dense pine forest
  State:  lost
object missing at the back of aisle 12
  State:  of
there is something missing somewhere
  State:  is
lost boy, 10 years old, wearing red winter clothing in a forest
  State:  in


#### Results of iteration 1
So this doesn't really work. It fails more than it succeeds to be honest. So to fix that, here are some observations:
1. The actual state verb isn't always the root so all verbs need to be considered
2. When there are multiple verbs in the sentence, the correct one seems to be one that serves as the "adjectival modifier"/"amod" to the subject

Based on these, the list of priorities I want to test are:
1. Be the root & a verb
2. Be the only verb
3. Be a verb that serves as a "adjectival modifier"/"amod"
4. Be a preposition

By implementing this list, and also adding in checks for "clausal modifier of noun"/"acl"s, I made the algorithm work for my test cases.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def parse(prompt):
    print(prompt)
    promptDoc = nlp(prompt)
    state = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
    if (state == ""):
        possibleState = []
        for token in promptDoc:
            if(token.pos_ == "VERB"):
                possibleState.append(token.text)
        if(len(possibleState) == 1):
            state = possibleState[0]
    if(state == ""):
        for token in promptDoc:
            if(token.pos_ == "VERB" and token.dep_ in ["amod", "acl"]):
                state = token.text
    if (state == ""):
        for token in promptDoc:
            if(token.pos_ == "ADP"):
                state = token.text
    print("  State: ", state)

parse("10 year old boy in red winter clothing lost in the dense forest")
parse("10 year old boy in red winter clothing")
# This didn't work: missing is not considered an adjectival modifier
# Must also include "clausal modifier of noun"/"acl"
# Including acl worked and fixed this
parse("small child wearing a blue jacket missing in the park")
parse("gray backpack lost in aisle C")
# This now identified the state as found which is probably what we want
parse("plastic elephant toy found in aisle B")
parse("lost child in the north section of the dense pine forest")
parse("object missing at the back of aisle 12")
# This one shouldn't work anyways but it actually does here (?)
parse("there is something missing somewhere")
parse("lost boy, 10 years old, wearing red winter clothing in a forest")


10 year old boy in red winter clothing lost in the dense forest
  State:  lost
10 year old boy in red winter clothing
  State:  in
small child wearing a blue jacket missing in the park
  State:  missing
gray backpack lost in aisle C
  State:  lost
plastic elephant toy found in aisle B
  State:  found
lost child in the north section of the dense pine forest
  State:  lost
object missing at the back of aisle 12
  State:  missing
there is something missing somewhere
  State:  is
lost boy, 10 years old, wearing red winter clothing in a forest
  State:  lost


## Identifying Descriptions
Now that the search object and its state can be determined, the algorithm needs to figure out what descriptions are given about it.

My first thought is to check every word that isn't the object. By looking at the test graphs above, that *seems* to be valid reasoning so I'm going to test that out and go from there. Step 1 is to condense the subject/state parsing functions into a singular algorithm so the description parsing part can use the previous results. That combined code and the tests for my first iteration is below.

In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")

results = {
    "object": "",
    "state": [],
    "descriptions": []
}

def findObject(promptDoc):
    obj = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence that isn't the object we want
            for tok in token.children:
                if(tok.pos_ == "NOUN" and tok.dep_ in ["nsubj", "nsubjpass", "dobj"]):
                    obj = tok.text
        elif(token.dep_ == "ROOT"):
            # We have found the noun root of the sentence
            obj = token.text
    results["object"] = obj

def findState(promptDoc):
    state = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
    if (state == ""):
        possibleState = []
        for token in promptDoc:
            if(token.pos_ == "VERB"):
                possibleState.append(token.text)
        if(len(possibleState) == 1):
            state = possibleState[0]
    if(state == ""):
        for token in promptDoc:
            if(token.pos_ == "VERB" and token.dep_ in ["amod", "acl"]):
                state = token.text
    if (state == ""):
        for token in promptDoc:
            if(token.pos_ == "ADP"):
                state = token.text
    results["state"].append(state)

def findDesc(promptDoc):
    desc = []
    for token in promptDoc:
        if(token.pos_ == "NOUN" and token.text != results["object"]):
            desc.append(token.text)
    results["descriptions"] = desc

def parse(prompt):
    results["object"] = ""
    results["state"] = []
    results["descriptions"] = []
    print(prompt)
    promptDoc = nlp(prompt)
    findObject(promptDoc)
    findState(promptDoc)
    findDesc(promptDoc)
    print(results["descriptions"])

parse("10 year old boy in red winter clothing lost in the dense forest")
parse("10 year old boy in red winter clothing")
parse("small child wearing a blue jacket missing in the park")
parse("gray backpack lost in aisle C")
parse("plastic elephant toy found in aisle B")
# This one didn't work at first so after testing, I added "dobj"/"direct object"s to the possible dependencies
parse("lost child in the north section of the dense pine forest")
parse("object missing at the back of aisle 12")
# This one shouldn't work anyways
parse("there is something missing somewhere")
parse("lost boy, 10 years old, wearing red winter clothing in a forest")


10 year old boy in red winter clothing lost in the dense forest
['year', 'winter', 'clothing', 'forest']
10 year old boy in red winter clothing
['year', 'winter', 'clothing']
small child wearing a blue jacket missing in the park
['jacket', 'park']
gray backpack lost in aisle C
['aisle']
plastic elephant toy found in aisle B
['elephant', 'aisle']
lost child in the north section of the dense pine forest
['north', 'section', 'pine', 'forest']
object missing at the back of aisle 12
['back', 'aisle']
there is something missing somewhere
[]
lost boy, 10 years old, wearing red winter clothing in a forest
['years', 'winter', 'clothing', 'forest']


### Results of iteration 1
This method of extracting descriptions isn't bad. It does miss things but it's a good start. I think the best next step is to go back to the dependency tree and try to connect tokens together into singular descriptions. 
For example, in the first test prompt, "winter" and "clothing" are detected as 2 separate descriptions but in reality, they should be one together. From the visual graph, I can see that "winter" is a *compound* and its head is "clothing".

Still considering dependencies, "red" is an amod/adjectival modifier to "winter" and "winter" is also its head. Based on these observations, I wrote the below algorithm to parse the tree.

In [39]:
import spacy

nlp = spacy.load("en_core_web_sm")

results = {
    "object": "",
    "state": [],
    "descriptions": []
}

def findObject(promptDoc):
    obj = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence that isn't the object we want
            for tok in token.children:
                if(tok.pos_ == "NOUN" and tok.dep_ in ["nsubj", "nsubjpass", "dobj"]):
                    obj = tok.text
        elif(token.dep_ == "ROOT"):
            # We have found the noun root of the sentence
            obj = token.text
    results["object"] = obj

def findState(promptDoc):
    state = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
    if (state == ""):
        possibleState = []
        for token in promptDoc:
            if(token.pos_ == "VERB"):
                possibleState.append(token.text)
        if(len(possibleState) == 1):
            state = possibleState[0]
    if(state == ""):
        for token in promptDoc:
            if(token.pos_ == "VERB" and token.dep_ in ["amod", "acl"]):
                state = token.text
    if (state == ""):
        for token in promptDoc:
            if(token.pos_ == "ADP"):
                state = token.text
    results["state"].append(state)

def findDesc(promptDoc):
    desc = []
    for token in promptDoc:
        curr = []
        if(token.dep_ == "amod"):
            curr.append(token.text)
            if(token.head.pos_ == "NOUN" and token.text != results["object"]):
                curr.append(token.head.text)
                if(token.head.dep_ == "compound" and token.head.head.pos_ == "NOUN" and token.head.head.text != results["object"]):
                    curr.append(token.head.head.text)
        if(len(curr) > 0): desc.append(" ".join(desc for desc in curr if desc != results["object"]))
    results["descriptions"] = desc

def parse(prompt):
    results["object"] = ""
    results["state"] = []
    results["descriptions"] = []
    print(prompt)
    promptDoc = nlp(prompt)
    findObject(promptDoc)
    findState(promptDoc)
    findDesc(promptDoc)
    print(results)

parse("10 year old boy in red winter clothing lost in the dense forest")
parse("10 year old boy in red winter clothing")
parse("small child wearing a blue jacket missing in the park")
parse("gray backpack lost in aisle C")
parse("plastic elephant toy found in aisle B")
# This one didn't work at first so after testing, I added "dobj"/"direct object"s to the possible dependencies
parse("lost child in the north section of the dense pine forest")
parse("object missing at the back of aisle 12")
# This one shouldn't work anyways
parse("there is something missing somewhere")
parse("lost boy, 10 years old, wearing red winter clothing in a forest")


10 year old boy in red winter clothing lost in the dense forest
{'object': 'boy', 'state': ['lost'], 'descriptions': ['old', 'red winter clothing', 'dense forest']}
10 year old boy in red winter clothing
{'object': 'boy', 'state': ['in'], 'descriptions': ['old', 'red winter clothing']}
small child wearing a blue jacket missing in the park
{'object': 'child', 'state': ['missing'], 'descriptions': ['small', 'blue jacket']}
gray backpack lost in aisle C
{'object': 'backpack', 'state': ['lost'], 'descriptions': ['gray']}
plastic elephant toy found in aisle B
{'object': 'toy', 'state': ['found'], 'descriptions': []}
lost child in the north section of the dense pine forest
{'object': 'child', 'state': ['lost'], 'descriptions': ['dense forest']}
object missing at the back of aisle 12
{'object': 'object', 'state': ['missing'], 'descriptions': []}
there is something missing somewhere
{'object': '', 'state': ['is'], 'descriptions': []}
lost boy, 10 years old, wearing red winter clothing in a for

### Results of iteration 2
This works much much better. Some details still get lost, however. Things like the specific age of the object (test case 1 for example) and other descriptors like "plastic elephant" are lost.

Additionally, descriptors that don't describe the search object do get found. For example in test 6, "dense forrest" is matched. However, this describes the context of the search prompt rather than the object itself which is an issue.

To fix the age issue, I believe just adding nummod to the list of checked modifiers should work. Additionally, compounds themselves should also be checked (at least that's what it looks like for the "plastic elephant" case).

These changes are implemented below.

In [71]:
import spacy

nlp = spacy.load("en_core_web_sm")

results = {
    "object": "",
    "state": [],
    "descriptions": []
}

def findObject(promptDoc):
    obj = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence that isn't the object we want
            for tok in token.children:
                if(tok.pos_ == "NOUN" and tok.dep_ in ["nsubj", "nsubjpass", "dobj"]):
                    obj = tok.text
        elif(token.dep_ == "ROOT"):
            # We have found the noun root of the sentence
            obj = token.text
    results["object"] = obj

def findState(promptDoc):
    state = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
    if (state == ""):
        possibleState = []
        for token in promptDoc:
            if(token.pos_ == "VERB"):
                possibleState.append(token.text)
        if(len(possibleState) == 1):
            state = possibleState[0]
    if(state == ""):
        for token in promptDoc:
            if(token.pos_ == "VERB" and token.dep_ in ["amod", "acl"]):
                state = token.text
    if (state == ""):
        for token in promptDoc:
            if(token.pos_ == "ADP"):
                state = token.text
    results["state"].append(state)

def findDesc(promptDoc):
    desc = []
    for token in promptDoc:
        curr = []
        if((token.dep_ in ["amod", "nummod"] or (token.dep_ == "compound")) and len(list(token.children)) == 0):
            curr.append(token.text)
            if(token.head.pos_ == "NOUN" and token.text != results["object"]):
                curr.append(token.head.text)
                if(token.head.dep_ in ["compound", "npadvmod"] and token.head.head.pos_ in ["NOUN", "ADJ"] and token.head.head.text != results["object"]):
                    curr.append(token.head.head.text)
        if(curr != []): 
            desc.append(" ".join(desc for desc in curr if desc != results["object"]))
    results["descriptions"] = desc

def parse(prompt):
    results["object"] = ""
    results["state"] = []
    results["descriptions"] = []
    print(prompt)
    promptDoc = nlp(prompt)
    findObject(promptDoc)
    findState(promptDoc)
    findDesc(promptDoc)
    print(results)

parse("10 year old boy in red winter clothing lost in the dense forest")
parse("10 year old boy in red winter clothing")
parse("small child wearing a blue jacket missing in the park")
parse("gray backpack lost in aisle C")
parse("plastic elephant toy found in aisle B")
# This one didn't work at first so after testing, I added "dobj"/"direct object"s to the possible dependencies
parse("lost child in the north section of the dense pine forest")
parse("object missing at the back of aisle 12")
# This one shouldn't work anyways
parse("there is something missing somewhere")
parse("lost boy, 10 years old, wearing red winter clothing in a forest")


10 year old boy in red winter clothing lost in the dense forest
{'object': 'boy', 'state': ['lost'], 'descriptions': ['10 year old', 'red winter clothing', 'dense forest']}
10 year old boy in red winter clothing
{'object': 'boy', 'state': ['in'], 'descriptions': ['10 year old', 'red winter clothing']}
small child wearing a blue jacket missing in the park
{'object': 'child', 'state': ['missing'], 'descriptions': ['small', 'blue jacket']}
gray backpack lost in aisle C
{'object': 'backpack', 'state': ['lost'], 'descriptions': ['gray', 'aisle']}
plastic elephant toy found in aisle B
{'object': 'toy', 'state': ['found'], 'descriptions': ['plastic', 'elephant', 'aisle']}
lost child in the north section of the dense pine forest
{'object': 'child', 'state': ['lost'], 'descriptions': ['north section', 'dense forest', 'pine forest']}
object missing at the back of aisle 12
{'object': 'object', 'state': ['missing'], 'descriptions': ['12 aisle']}
there is something missing somewhere
{'object': '', 

### Results of iteration 3
With the last update, it works great! Everything that I would expect to get marked as a description does get picked up. Additional information like "dense forrest" still get counted though which needs updating.

But before that, I think I need to get the context parsed properly so that's going to be the next step and then we can come back to this.